In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
import re

## deterministic guardrails - true or false 
def deterministic_guardrails(text:str) -> bool:
    """Returns True if content is blocked"""
    banned_keywords=["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do i hack into a database?",
    "What is the capital of france?",
    "Explain how malware spread across the computer?"
]

print("=== Deterministic approach ===")
for inp in test_inputs:
    blocked = deterministic_guardrails(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}:{inp}")

=== Deterministic approach ===
Blocked:How do i hack into a database?
Allowed:What is the capital of france?
Blocked:Explain how malware spread across the computer?


In [4]:
### Model based approach 

from langchain_groq import ChatGroq

### calling out the model 
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety, falling back to deterministic rules if needed."""
    prompt = f"""Is the following user input safe to process?
    Reply with only 'SAFE' or 'UNSAFE'

Input: {text}"""

    candidate_models = [
        "llama-3.3-70b-versatile",
        "llama-3.1-8b-instant",
    ]

    last_error = None
    for model_name in candidate_models:
        try:
            model = ChatGroq(model=model_name, temperature=0)
            result = model.invoke([{"role": "user", "content": prompt}])
            return result.content.strip()
        except Exception as exc:
            last_error = exc

    if deterministic_guardrails(text):
        return "UNSAFE"
    return "SAFE"

print("=== Model_Based Guardrails ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status} : {inp}")


=== Model_Based Guardrails ===
UNSAFE : How do i hack into a database?
SAFE : What is the capital of france?
SAFE : Explain how malware spread across the computer?


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# Define system prompt to guide model when receiving masked/redacted info
system_prompt = (
    "You are a helpful customer service assistant. "
    "If the user provides credit card information or email, they will be masked or redacted by the middleware. "
    "Use the masked information (like the card ending in 5100) to call the customer_lookup tool. "
    "Once you retrieve the record, reply to the user exactly like this: "
    "'I found the customer record associated with the card ending in 5100. How may I assist you further with this information?'"
)

# Create agent with PII Middleware and System Prompt
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[customer_lookup],
    system_prompt=system_prompt,
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [6]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I found the customer record associated with the card ending in 5100. How may I assist you further with this information?


In [7]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='1475858a-7719-4d3b-9347-28344fadc4e6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'z1m5xx993', 'function': {'arguments': '{"query":"email: [REDACTED_EMAIL] and card ending in: 5100"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 326, 'total_tokens': 357, 'completion_time': 0.050796732, 'completion_tokens_details': None, 'prompt_time': 0.021944418, 'prompt_tokens_details': None, 'queue_time': 0.300906683, 'total_time': 0.07274115}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2e35-d2be-7a32-a1b0-7bfc315b2783-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query

In [8]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"Blocked as expected:{e}")

Blocked as expected:Detected 1 instance(s) of api_key in text content


In [9]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='1475858a-7719-4d3b-9347-28344fadc4e6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'z1m5xx993', 'function': {'arguments': '{"query":"email: [REDACTED_EMAIL] and card ending in: 5100"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 326, 'total_tokens': 357, 'completion_time': 0.050796732, 'completion_tokens_details': None, 'prompt_time': 0.021944418, 'prompt_tokens_details': None, 'queue_time': 0.300906683, 'total_time': 0.07274115}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2e35-d2be-7a32-a1b0-7bfc315b2783-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Define system prompt to restrict tool hallucination and control flow
system_prompt = (
    "You are a helpful office assistant. "
    "If you need to search the web, you must use the `search_web` tool. "
    "Do not attempt to call any tools that are not provided in your tool list (such as `brave_search`). "
    "Once you have successfully executed a tool (like sending an email or deleting records), "
    "simply report the action result back to the user and stop. Do not invoke additional tools or do extra research."
)

# Create agent with HITL middleware, System Prompt, and llama-3.3-70b-versatile
hitl_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_web, send_email, delete_records],
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [11]:
## Test the agent
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='45706a5b-cac9-46aa-8b53-9e73fb6adda5'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rggrx6xxd', 'function': {'arguments': '{"body":"Please find the Q4 results attached.","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 433, 'total_tokens': 471, 'completion_time': 0.094332422, 'completion_tokens_details': None, 'prompt_time': 0.022174757, 'prompt_tokens_details': None, 'queue_time': 0.161738481, 'total_time': 0.116507179}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2e35-d717-7bc2-8a79-ec234dc6f16f-0', tool

In [12]:
## Approval by human 
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
Email sent to team@company.com with subject: Q4 Results


In [13]:
## custom middleware - before agent hook

from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f" Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Define system prompt to avoid unnecessary tool calls and potential formatting errors
system_prompt = (
    "You are a helpful assistant. Answer the user's questions directly using your knowledge base. "
    "Do not use the `search_tool` unless you are specifically asked to search or do not know the answer."
)

# Create agent with content filter and system prompt
filtered_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_tool],
    system_prompt=system_prompt,
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [14]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perform tasks without being explicitly programmed. It allows systems to learn from data, identify patterns, and make decisions or predictions based on that data. The goal of machine learning is to develop models that can improve their performance on a task over time, without being explicitly told how to do so. There are several types of machine learning, including supervised learning, unsupervised learning, and reinforcement learning, each with its own approach to learning from data.


In [15]:
## Test - 2 : unsafe request 

result = filtered_agent.invoke({
    "messages":[{
        "role":"user","content":"How do i hack into someone server?"
    }]
})

print("Unsafe request response:")
print(result["messages"][-1].content)

 Blocked — keyword detected: 'hack'
Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


In [16]:
### Custom Guardrail - After - agent hook

from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.runtime import Runtime


class SafetyGuardMiddleware(AgentMiddleware):
    """
    Model-based guardrails: use an LLM to evaluate the agent's response
    before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        self.safety_model = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime,
    ) -> dict[str, Any] | None:
        if not state.get("messages"):
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        response_text = last_message.content
        if not isinstance(response_text, str):
            response_text = str(response_text)

        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
        Respond with only "SAFE" or "UNSAFE".

        Response to evaluate:
        {response_text}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])
        verdict = result.content.strip().upper()

        if "UNSAFE" in verdict:
            print("Output flagged as UNSAFE - replacing with a safe fallback")
            last_message.content = (
                "I am unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general-purpose tool for the example."""
    return f"Tool result: {query}"


system_prompt = (
    "You are a helpful assistant. "
    "You may see a general tool, but do not use it unless the user explicitly asks for a tool-based action. "
    "For this notebook example, answer directly from your own knowledge and follow the guardrail behavior."
)

safe_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[general_tool],
    system_prompt=system_prompt,
    middleware=[SafetyGuardMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [17]:
## Testing output safety check

result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the capital of France?"}]
})
print("Response:")
print(result["messages"][-1].content)


Response:
The capital of France is Paris.


In [22]:
### Layered / combined guardrails

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware,HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query:str) -> str:
    """Search for information"""
    return f"Search results:{query}"

@tool
def send_email_tool(to:str,body:str)->str:
    """Send an email."""
    return f"Email send to {to}"

production_agent = create_agent(
    model = "groq:llama-3.1-8b-instant",
    tools=[send_email_tool,search_tool],
    middleware=[

        ## Layer 1 :- Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack","exploit","malware"]),

        ## Layer 2 :- PII redaction on input 
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),

        ## Layer 3:- Human in the loop middleware
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool":True,"search_tool":False},
        ),

        ## Layer 4 :- PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)
print("Production-grade agent with 5-layer guardrails created!")

Production-grade agent with 5-layer guardrails created!
